# 10 COMPLEX SQL QUERIES

In [12]:
import numpy as np
import pandas as pd
import duckdb

def sqldf(query):
    return duckdb.sql(query).df()

## DATASET

In [13]:
employees_data = {
    "emp_id": [
        1,2,3,4,5,
        6,7,8,9,10,
        11,12,13,14,15
    ],

    "name": [
        "Alice","Bob","Charlie","David","Eva",
        "Frank","Grace","Henry","Ivy","Jack",
        "Karen","Leo","Mona","Nathan","Olivia"
    ],

    "dept_id": [
        101,102,101,103,104,
        102,105,101,104,103,
        102,105,101,103,104
    ],

    "salary": [
        70000,50000,80000,45000,60000,
        52000,75000,90000,65000,47000,
        54000,72000,83000,46000,62000
    ],

    "experience": [
        5,3,7,2,4,
        3,6,8,5,2,
        4,7,9,2,5
    ],

    "city": [
        "Chennai","Delhi","Mumbai","Bangalore","Chennai",
        "Delhi","Mumbai","Pune","Hyderabad","Bangalore",
        "Delhi","Mumbai","Pune","Bangalore","Hyderabad"
    ]
}

employees = pd.DataFrame(employees_data)
employees

,emp_id,name,dept_id,salary,experience,city
0,1,Alice,101,70000,5,Chennai
1,2,Bob,102,50000,3,Delhi
2,3,Charlie,101,80000,7,Mumbai
3,4,David,103,45000,2,Bangalore
4,5,Eva,104,60000,4,Chennai
5,6,Frank,102,52000,3,Delhi
6,7,Grace,105,75000,6,Mumbai
7,8,Henry,101,90000,8,Pune
8,9,Ivy,104,65000,5,Hyderabad
9,10,Jack,103,47000,2,Bangalore


In [14]:
departments_data = {
    "dept_id": [101,102,103,104,105],

    "dept_name": [
        "Engineering",
        "HR",
        "Sales",
        "Marketing",
        "Finance"
    ],

    "location": [
        "Chennai",
        "Delhi",
        "Mumbai",
        "Bangalore",
        "Pune"
    ],

    "budget": [
        500000,
        200000,
        350000,
        300000,
        250000
    ]
}

departments = pd.DataFrame(departments_data)
departments

,dept_id,dept_name,location,budget
0,101,Engineering,Chennai,500000
1,102,HR,Delhi,200000
2,103,Sales,Mumbai,350000
3,104,Marketing,Bangalore,300000
4,105,Finance,Pune,250000


In [15]:
projects_data = {
    "project_id": [
        201,202,203,204,205,
        206,207,208,209,210
    ],

    "emp_id": [
        1,3,5,7,9,
        2,4,6,8,10
    ],

    "project_name": [
        "AI Platform",
        "Cloud Migration",
        "CRM Upgrade",
        "Fraud Detection",
        "Data Warehouse",
        "HR Analytics",
        "Customer App",
        "Automation Tool",
        "Ad Campaign",
        "Market Analysis"
    ],

    "project_budget": [
        120000,
        150000,
        90000,
        110000,
        130000,
        80000,
        95000,
        85000,
        100000,
        75000
    ],

    "duration_months": [
        12,10,8,9,11,
        6,7,8,6,5
    ]
}

projects = pd.DataFrame(projects_data)
projects

,project_id,emp_id,project_name,project_budget,duration_months
0,201,1,AI Platform,120000,12
1,202,3,Cloud Migration,150000,10
2,203,5,CRM Upgrade,90000,8
3,204,7,Fraud Detection,110000,9
4,205,9,Data Warehouse,130000,11
5,206,2,HR Analytics,80000,6
6,207,4,Customer App,95000,7
7,208,6,Automation Tool,85000,8
8,209,8,Ad Campaign,100000,6
9,210,10,Market Analysis,75000,5


### Queries:

In [ ]:
#1)
#Departments whose average employee salary exceeds 60k.
sql_query = """
SELECT d.dept_name,
       AVG(e.salary) AS avg_salary,
       COUNT(e.emp_id) AS num_employees
FROM employees e
JOIN departments d
ON e.dept_id = d.dept_id
GROUP BY d.dept_name
HAVING AVG(e.salary) > 60000
ORDER BY avg_salary DESC;
"""

result_df = sqldf(sql_query)
print(result_df)

     dept_name    avg_salary  num_employees
0  Engineering  80750.000000              4
1      Finance  73500.000000              2
2    Marketing  62333.333333              3


In [ ]:
#2) Rank employees within each department by salary.
sql_query = """
SELECT e.name,
       d.dept_name,
       e.salary,
       RANK() OVER(
           PARTITION BY d.dept_name
           ORDER BY e.salary DESC
       ) AS dept_salary_rank
FROM employees e
INNER JOIN departments d
ON e.dept_id = d.dept_id;
"""
result_df = sqldf(sql_query)
print(result_df)


       name    dept_name  salary  dept_salary_rank
0     Grace      Finance   75000                 1
1       Leo      Finance   72000                 2
2     Karen           HR   54000                 1
3     Frank           HR   52000                 2
4       Bob           HR   50000                 3
5      Jack        Sales   47000                 1
6    Nathan        Sales   46000                 2
7     David        Sales   45000                 3
8     Henry  Engineering   90000                 1
9      Mona  Engineering   83000                 2
10  Charlie  Engineering   80000                 3
11    Alice  Engineering   70000                 4
12      Ivy    Marketing   65000                 1
13   Olivia    Marketing   62000                 2
14      Eva    Marketing   60000                 3


In [31]:
#3) Employees earning above their department average.
sql_query = """
SELECT e.name, e.salary, e.dept_id
FROM employees e
WHERE e.salary >
(
    SELECT AVG(salary)
    FROM employees
    WHERE dept_id = e.dept_id
)
"""
result_df = sqldf(sql_query)
print(result_df)

    name  salary  dept_id
0  Grace   75000      105
1  Henry   90000      101
2    Ivy   65000      104
3   Jack   47000      103
4  Karen   54000      102
5   Mona   83000      101


In [35]:
#4) Employees working in departments whose budget is above average.
sql_query = """
SELECT e.name, d.dept_name, d.budget
FROM employees e
INNER JOIN departments d
ON e.dept_id = d.dept_id
WHERE d.budget >
(
    SELECT AVG(budget)
    FROM departments
)
"""
result_df = sqldf(sql_query)
print(result_df)

      name    dept_name  budget
0    Alice  Engineering  500000
1  Charlie  Engineering  500000
2    David        Sales  350000
3    Henry  Engineering  500000
4     Jack        Sales  350000
5     Mona  Engineering  500000
6   Nathan        Sales  350000


In [36]:
# 5)Categorize employees by salary level.
sql_query = """
SELECT e.name,
       d.dept_name,
       e.salary,
       CASE
           WHEN e.salary >= 80000 THEN 'High'
           WHEN e.salary >= 60000 THEN 'Medium'
           ELSE 'Low'
       END AS salary_category
FROM employees e
INNER JOIN departments d
ON e.dept_id = d.dept_id
ORDER BY e.salary DESC
"""
result_df = sqldf(sql_query)
print(result_df)

       name    dept_name  salary salary_category
0     Henry  Engineering   90000            High
1      Mona  Engineering   83000            High
2   Charlie  Engineering   80000            High
3     Grace      Finance   75000          Medium
4       Leo      Finance   72000          Medium
5     Alice  Engineering   70000          Medium
6       Ivy    Marketing   65000          Medium
7    Olivia    Marketing   62000          Medium
8       Eva    Marketing   60000          Medium
9     Karen           HR   54000             Low
10    Frank           HR   52000             Low
11      Bob           HR   50000             Low
12     Jack        Sales   47000             Low
13   Nathan        Sales   46000             Low
14    David        Sales   45000             Low


In [38]:
# 6) Divide employees into 4 salary quartiles.
sql_query = """
SELECT e.name,
       d.dept_name,
       e.salary,
       NTILE(4) OVER(
           ORDER BY e.salary
       ) AS salary_quartile
FROM employees e
JOIN departments d
ON e.dept_id = d.dept_id
"""
result_df = sqldf(sql_query)
print(result_df)

       name    dept_name  salary  salary_quartile
0     David        Sales   45000                1
1    Nathan        Sales   46000                1
2      Jack        Sales   47000                1
3       Bob           HR   50000                1
4     Frank           HR   52000                2
5     Karen           HR   54000                2
6       Eva    Marketing   60000                2
7    Olivia    Marketing   62000                2
8       Ivy    Marketing   65000                3
9     Alice  Engineering   70000                3
10      Leo      Finance   72000                3
11    Grace      Finance   75000                3
12  Charlie  Engineering   80000                4
13     Mona  Engineering   83000                4
14    Henry  Engineering   90000                4


In [39]:
# 7) 
sql_query = """
SELECT
    e.name,
    d.dept_name,
    e.salary,

    -- Salary of employee just above in department ranking
    LAG(e.salary) OVER (
        PARTITION BY d.dept_name
        ORDER BY e.salary DESC
    ) AS higher_salary,

    -- Salary of employee just below in department ranking
    LEAD(e.salary) OVER (
        PARTITION BY d.dept_name
        ORDER BY e.salary DESC
    ) AS lower_salary,

    -- Highest paid employee in department
    FIRST_VALUE(e.name) OVER (
        PARTITION BY d.dept_name
        ORDER BY e.salary DESC
    ) AS top_employee,

    -- Lowest paid employee in department
    LAST_VALUE(e.name) OVER (
        PARTITION BY d.dept_name
        ORDER BY e.salary DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS lowest_paid_employee

FROM employees e
JOIN departments d
ON e.dept_id = d.dept_id

ORDER BY d.dept_name, e.salary DESC
"""
result_df = sqldf(sql_query)
print(result_df)

       name    dept_name  salary  higher_salary  lower_salary top_employee  \
0     Henry  Engineering   90000           <NA>         83000        Henry   
1      Mona  Engineering   83000          90000         80000        Henry   
2   Charlie  Engineering   80000          83000         70000        Henry   
3     Alice  Engineering   70000          80000          <NA>        Henry   
4     Grace      Finance   75000           <NA>         72000        Grace   
5       Leo      Finance   72000          75000          <NA>        Grace   
6     Karen           HR   54000           <NA>         52000        Karen   
7     Frank           HR   52000          54000         50000        Karen   
8       Bob           HR   50000          52000          <NA>        Karen   
9       Ivy    Marketing   65000           <NA>         62000          Ivy   
10   Olivia    Marketing   62000          65000         60000          Ivy   
11      Eva    Marketing   60000          62000          <NA>   

In [41]:

data = {
'CustomerID': [1, 2, 3, 4, 5],
'Name': [' john ', 'MARY JONES', 'peter.k ', ' anna ', 'robert '],
'ProductCode': ['B-210-L', 'C-415-XL', 'D-510-S', 'B-210-M ', 'E-620-M'],
'Email': [
' John@mail.com ',
'MARY@work.org ',
'peterK@web.net ',
' ANNA@corp.com',
'robert@home.io '
]
}

customer_data = pd.DataFrame(data)

customer_data

,CustomerID,Name,ProductCode,Email
0,1,john,B-210-L,John@mail.com
1,2,MARY JONES,C-415-XL,MARY@work.org
2,3,peter.k,D-510-S,peterK@web.net
3,4,anna,B-210-M,ANNA@corp.com
4,5,robert,E-620-M,robert@home.io


In [42]:
# 8) Data cleaning
sql_query = """
SELECT
    CustomerID,

    -- Clean and standardize name
    UPPER(TRIM(Name)) AS Cleaned_Name,

    -- Clean product code
    REPLACE(UPPER(TRIM(ProductCode)), '-', '') AS Cleaned_ProductCode,

    -- Clean email
    LOWER(TRIM(Email)) AS Cleaned_Email,

    -- Extract size category
    CASE
        WHEN ProductCode LIKE '%XL%' THEN 'Extra Large'
        WHEN ProductCode LIKE '%L%' THEN 'Large'
        WHEN ProductCode LIKE '%M%' THEN 'Medium'
        WHEN ProductCode LIKE '%S%' THEN 'Small'
        ELSE 'Unknown'
    END AS Product_Size

FROM customer_data
"""
result_df = sqldf(sql_query)
print(result_df)

   CustomerID Cleaned_Name Cleaned_ProductCode   Cleaned_Email Product_Size
0           1         JOHN               B210L   john@mail.com        Large
1           2   MARY JONES              C415XL   mary@work.org  Extra Large
2           3      PETER.K               D510S  peterk@web.net        Small
3           4         ANNA               B210M   anna@corp.com       Medium
4           5       ROBERT               E620M  robert@home.io       Medium


In [ ]:
# 9) Employees working in departments that have more than 3 employees
sql_query = """
SELECT name, dept_id
FROM employees
WHERE dept_id IN
(
    SELECT dept_id
    FROM employees
    GROUP BY dept_id
    HAVING COUNT(*) > 3
)
"""

result_df = sqldf(sql_query)
print(result_df)

      name  dept_id
0    Alice      101
1  Charlie      101
2    Henry      101
3     Mona      101


In [49]:
data = {
'CustomerID': [1, 2, 3, 4, 5, 6],
'CustomerName': ['John', 'Mary', 'Peter', 'Anna', 'Robert', 'David'],
'ProductID': [101, 102, 103, 104, 105, 106],
'Quantity': [5, np.nan, 10, 0, 7, np.nan],
'Price': [200, 150, np.nan, 300, 250, 180]
}

customer_sales = pd.DataFrame(data)
customer_sales

,CustomerID,CustomerName,ProductID,Quantity,Price
0,1,John,101,5.0,200.0
1,2,Mary,102,NaN,150.0
2,3,Peter,103,10.0,NaN
3,4,Anna,104,0.0,300.0
4,5,Robert,105,7.0,250.0
5,6,David,106,NaN,180.0


In [51]:
sql_query = """
SELECT
    CustomerID,
    CustomerName,
    ProductID,

    -- Treat quantity 0 as NULL
    NULLIF(Quantity,0) AS Cleaned_Quantity,

    -- Replace NULL quantity with -1
    COALESCE(Cleaned_Quantity, -1) AS Final_Quantity,

    -- Replace NULL price with average price
    COALESCE(Price,
        (SELECT AVG(Price) FROM customer_sales)
    ) AS Cleaned_Price

FROM customer_sales
"""

result_df = sqldf(sql_query)
print(result_df)

   CustomerID CustomerName  ProductID  Cleaned_Quantity  Final_Quantity  \
0           1         John        101               5.0             5.0   
1           2         Mary        102               NaN            -1.0   
2           3        Peter        103              10.0            10.0   
3           4         Anna        104               NaN            -1.0   
4           5       Robert        105               7.0             7.0   
5           6        David        106               NaN            -1.0   

   Cleaned_Price  
0          200.0  
1          150.0  
2          216.0  
3          300.0  
4          250.0  
5          180.0  
